# Medicine Recommendation System – EDA & Model Analysis
**M. Ali Sanwal| bc240440384mas@vu.edu.pk**
**CS619 | Spring 2026 | Supervisor: Dr. Mushtaq Hussain**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import json, os, warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
%matplotlib inline

## 1. Load Data

In [ ]:
# Generate dataset first if not present
if not os.path.exists('../data/patient_records.csv'):
    os.chdir('..')
    exec(open('data/generate_dataset.py').read())
    os.chdir('notebooks')

df = pd.read_csv('../data/patient_records.csv')
print(f'Shape: {df.shape}')
df.head()

## 2. Basic Statistics

In [ ]:
print('Diseases:', df['disease'].nunique())
print('Medicines:', df['primary_medicine'].nunique())
print('Records:', len(df))
df[['age','severity','symptom_count']].describe()

## 3. Disease Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(14,5))
disease_counts = df['disease'].value_counts()
sns.barplot(x=disease_counts.index, y=disease_counts.values, ax=ax, palette='Blues_d')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
ax.set_title('Records per Disease', fontsize=14)
plt.tight_layout(); plt.show()

## 4. Age Distribution by Disease

In [ ]:
fig = px.box(df, x='disease', y='age', color='disease',
             title='Age Distribution by Disease')
fig.update_layout(xaxis_tickangle=-45, showlegend=False)
fig.show()

## 5. Symptom Co-occurrence Heatmap

In [ ]:
sym_cols = [c for c in df.columns if c.startswith('sym_')]
sym_short = [c.replace('sym_','').replace('_',' ') for c in sym_cols[:20]]
corr = df[sym_cols[:20]].corr()
corr.columns = sym_short; corr.index = sym_short

fig, ax = plt.subplots(figsize=(12,10))
sns.heatmap(corr, annot=False, cmap='coolwarm', ax=ax)
ax.set_title('Symptom Co-occurrence Correlation')
plt.tight_layout(); plt.show()

## 6. Model Training

In [ ]:
import sys; sys.path.insert(0,'..')
from models.train_models import main as train_main
summary = train_main(data_path='../data/patient_records.csv', output_dir='../models')

## 7. Model Performance Comparison

In [ ]:
disease_df = pd.DataFrame(summary['disease_results']).T
fig, axes = plt.subplots(1,2,figsize=(14,5))

disease_df['test_accuracy'].sort_values().plot(kind='barh', ax=axes[0],
    color='steelblue', title='Disease Prediction – Test Accuracy')

med_df = pd.DataFrame(summary['medicine_results']).T
med_df['f1_score'].sort_values().plot(kind='barh', ax=axes[1],
    color='seagreen', title='Medicine Prediction – F1 Score')

plt.tight_layout(); plt.show()

## 8. Test the Recommender

In [ ]:
from src.recommender import MedicineRecommender
rec = MedicineRecommender(model_dir='../models', metadata_path='../data/metadata.json')

result = rec.recommend(
    symptoms=['headache', 'high fever', 'body ache', 'fatigue'],
    age=32, gender='Female', severity=2
)

print('Predicted Disease:', result['predicted_diseases'][0]['disease'])
print('\nRecommended Medicines:')
for m in result['recommended_medicines']:
    print(f"  - {m['medicine']} ({m['source']})")